# K513 · Week 6 Homework
## Model Evaluation

An insurer wants to know which policies are likely to make a claim this year, so the claims team can look at them before the claim arrives. You will build three models and work out which number to put in front of the head of claims.

**Nothing typed into this notebook is collected.** Every written answer goes into the Canvas quiz **Week 6 Homework - Model Evaluation**. An answer left here scores zero.

---
### Before you type anything

**File → Save a copy in Drive.**

This notebook is read-only for you. You can type into it and run it and it will look completely
normal, but nothing you do will be saved. Save your own copy first, every time.

---

### Using AI in this notebook

Gemini is built into Colab and you are welcome to use it here. Two things worth knowing:

- It does not know which columns you have or what we covered in class. Whatever it writes, you own.
- The most useful thing you can ask it is **"explain what this line does"** — not "write it for me".


One more, specific to this assignment: an AI will tell you an 88% accuracy is good. It cannot see that 88% of your rows are the negative class, because that is a fact about your data rather than about your code.

---

### Turn off Unwanted AI Assistance

AI-powered coding completion is turned on by default. It is convenient but does not give you a chance
to think and learn. Turning it off helps you learn. You can always turn it back on when needed.
- Tools → settings → AI Assistance → Uncheck "Show AI-powered inline code completions"
- Tools → settings → Uncheck "Show context-powered code completions"

---

### How to run a cell

Click on a cell, then press **Shift + Enter**. That runs it and moves you to the next one.

---

## How this is submitted

| | |
|---|---|
| **Where the answers go** | the Canvas quiz **Week 6 Homework - Model Evaluation** |
| **What you paste into it** | your written answers, and one link to this notebook |
| **The link** | Colab → **Share** → **General access** → **Anyone with the link** |

Check the link opens in a private browsing window before you paste it. A link nobody else can
open scores nothing.

**100 points.** Notebook link 5 · (a) 10 · (b) 10 · (c) 20 · (d) 15 · (e) 20 · Analyst's Note 20.

---

## The data

`AutoClaim.csv` — 20,000 motor policies.

| Column | |
|---|---|
| `Premium USD` | what the policyholder pays |
| `Car Power` | engine power rating |
| `AgeClass` | `Young` or `Elder` |
| `Gender` | `F` or `M` |
| `Rating Class` | `A` to `D` |
| `Branch`, `Zone`, `Region`, `City` | where the policy was written |
| `Claim` | **the target.** `Y` if the policyholder made a claim, `N` if not |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (confusion_matrix, accuracy_score, recall_score,
                             precision_score, classification_report,
                             roc_curve, auc, roc_auc_score)

CLAIM_URL = "https://raw.githubusercontent.com/jl-uscn/k513-data/main/AutoClaim.csv"
SEED = 42

In [ ]:
claim_df = pd.read_csv(CLAIM_URL)
print("Rows and columns:", claim_df.shape)
claim_df.head()

---

## (a) The baseline — 10 points

Build the baseline model, and report its accuracy on the test set. Fill in the blanks. You can ask AI what model to use to create a baseline for a classification model.

> ✏️ **Answer in Canvas** — *Week 6 Homework - Model Evaluation*, question **(a)**.
> Report the accuracy, say what the model predicts for every policy, and explain in one or two
> sentences why the accuracy is as high as it is.


In [ ]:
y = (claim_df['Claim'] == 'Y').astype(int)
X = claim_df.drop(columns=['Sample #', 'Claim'])

numeric_features = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
categorical_features = [c for c in X.columns if c not in numeric_features]

print("numeric    :", numeric_features)
print("categorical:", categorical_features)
print("\nClaim rate:", round(y.mean(), 4))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, _______)

baseline = ____(strategy='most_frequent').fit(X_train, y_train)
y_pred_baseline = baseline.predict(X_test)

print("Test accuracy:", round(accuracy_score(y_test, y_pred_baseline), 4))
print("Policies it flags as a claim:", int(y_pred_baseline.sum()))

---

## (b) Two real models — 10 points

A logistic regression and a decision tree, both at the default cut of 0.5.

> ✏️ **Answer in Canvas** — *Week 6 Homework - Model Evaluation*, question **(b)**.
> For each model, report the four counts from its confusion matrix **in words** — how many
> policies fell into each cell, described as policies rather than as "true positive".

In [ ]:
preprocessor_scaled = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(drop='if_binary', handle_unknown='ignore'), categorical_features)])

preprocessor_plain = ColumnTransformer(transformers=[
    ('num', 'passthrough', numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)])

logreg = Pipeline(steps=[
    ('preprocessor', preprocessor_scaled),
    ('classifier', LogisticRegression(max_iter=1000, random_state=SEED))]).fit(X_train, y_train)

tree = Pipeline(steps=[
    ('preprocessor', preprocessor_plain),
    ('classifier', DecisionTreeClassifier(max_depth=3, random_state=SEED))]).fit(X_train, y_train)

print("fitted")

In [ ]:
for name, model in [('Baseline     ', baseline),
                    ('Logistic     ', logreg),
                    ('Decision tree', tree)]:
    y_pred = model.predict(X_test)
    cm = ____(y_test, y_pred)
    print(f"{name}  train {model.score(X_train, y_train):.4f}   "
          f"test {accuracy_score(y_test, y_pred):.4f}   "
          f"flags {int(y_pred.sum()):>4} policies")
    print(cm, "\n")

---

## (c) One of your models found nothing — 20 points

Look at the row above for the decision tree. Then run the cells below.

The second one is the one to sit with. A classifier does not hand you a label — it hands you a
**probability for the positive class**, and the label is only what happens after that number meets
a cut. So go and get the numbers: it sorts every test policy by its probability of a claim,
highest first, and shows you the **top 50 from each model side by side** — with what
actually happened to each of those policies beside its score.

> ✏️ **Answer in Canvas** — *Week 6 Homework - Model Evaluation*, question **(c)**.
>
> 1. Which measure catches the tree, and which measures do not? Say why each behaves the way it
>    does.
> 2. What is the single highest score the tree gives **any** policy? Now explain the zero.
> 3. The two models' AUCs are 0.743 and 0.726 — nearly the same. How, when one found 42 claims
>    and the other found none?


In [ ]:
rows = []
for name, model in [('Baseline', baseline), ('Logistic regression', logreg),
                    ('Decision tree', tree)]:
    y_pred = model.predict(X_test)
    probs = model._________(X_test)[:, 1]
    rows.append({'model': name,
                 'accuracy': accuracy_score(y_test, y_pred),
                 'recall': recall_score(y_test, y_pred, zero_division=0),
                 'precision': precision_score(y_test, y_pred, zero_division=0),
                 'AUC': roc_auc_score(y_test, probs)})

pd.DataFrame(rows).round(4)

In [ ]:
# The positive class is column 1. Column 0 is the probability of *no* claim.
scores_logreg = logreg.predict_proba(X_test)[:, 1]
scores_tree   = tree.predict_proba(X_test)[:, 1]

def top_50(scores):
    """The 50 highest-scoring policies for one model, in order, with what
    actually happened to each. argsort() sorts upwards, so negate to flip it."""
    order = np.argsort(-scores, kind='stable')[:50]
    return scores[order].round(4), np.where(y_test.values[order] == 1, 'YES', '-')

lr_score, lr_claimed = top_50(scores_logreg)
tr_score, tr_claimed = top_50(scores_tree)

print("Highest score any single policy got")
print(f"  logistic regression : {scores_logreg.max():.4f}")
print(f"  decision tree       : {scores_tree.max():.4f}")
print("\nThe cut both models are being judged at:  0.50")

pd.DataFrame({
    'rank': np.arange(1, 51),
    'logistic score': lr_score,
    'logistic: claimed?': lr_claimed,
    'tree score': tr_score,
    'tree: claimed?': tr_claimed,
})


One more. Every distinct score the tree is capable of producing — one per leaf — with how many
test policies land on it and how many of those really did claim. The claim rate across the whole
test set is 0.120.


In [ ]:
leaf = (pd.DataFrame({'score': scores_tree, 'claimed': y_test.values})
        .groupby('score')
        .agg(policies=('claimed', 'size'), real_claims=('claimed', 'sum'))
        .sort_index(ascending=False))
leaf['claim rate'] = (leaf['real_claims'] / leaf['policies']).round(3)
leaf


---

## (d) Move the cut — 15 points

Sweep the cut on the logistic model and report three settings.

> ✏️ **Answer in Canvas** — *Week 6 Homework - Model Evaluation*, question **(d)**.
> Report three cuts and what each does to recall and precision. For each of the three, name the
> step of *Steps to Evaluate a Model* you land on, and say why. **The five steps are reproduced
> in the Canvas question**, so you do not need to go and find the slide.

In [ ]:
probs_logreg = logreg.predict_proba(X_test)[:, ____]

rows = []
for cut in [0.05, 0.10, 0.12, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]:
    pred = (probs_logreg >= cut).astype(int)
    rows.append({'cut': cut,
                 'policies flagged': int(pred.sum()),
                 'real claims caught': int(((pred == 1) & (y_test == 1)).sum()),
                 'recall': recall_score(y_test, pred, zero_division=0),
                 'precision': precision_score(y_test, pred, zero_division=0),
                 'accuracy': accuracy_score(y_test, pred)})

pd.DataFrame(rows).round(3)

---

## (e) The claims team can look at 500 policies a month — 20 points

Not 2,000, not 72. Five hundred.

> ✏️ **Answer in Canvas** — *Week 6 Homework - Model Evaluation*, question **(e)**.
> Which cut would you use, and how many real claims does the team find? Compare that with picking
> 500 policies at random, and say what the model is worth in one sentence a manager would repeat.

In [ ]:
def top_n(probabilities, n, seed=0):
    """The n highest-scoring policies, ties at the cut broken at random."""
    rng = np.random.default_rng(seed)
    order = np.lexsort((rng.random(len(probabilities)), -probabilities))
    chosen = order[:n]
    return {'reviewed': n,
            'real claims found': int(y_test.values[chosen].sum()),
            'precision': round(float(y_test.values[chosen].mean()), 3)}

print("Ranked by the model:", top_n(probs_logreg, 500))
print("Picked at random   :", int(500 * y_test.mean()), "real claims")

Draw the ROC curve while you are here — you will want it for the Analyst's Note.

In [ ]:
plt.figure(figsize=(6, 5))
for model, name in [(logreg, 'Logistic regression'), (tree, 'Decision tree')]:
    probs = model.predict_proba(X_test)[:, 1]
    fpr, tpr, cuts = roc_curve(y_test, probs)
    plt.plot(fpr, tpr, label=f'{name}: AUC {auc(fpr, tpr):.3f}')
plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Knows nothing: 0.500')
plt.xlabel('false alarm rate')
plt.ylabel('recall')
plt.legend(loc='lower right')
plt.show()

---

## The Analyst's Note — 20 points

The head of claims runs the team. She has not taken this course and does not want to.

> ✏️ **Answer in Canvas** — *Week 6 Homework - Model Evaluation*, question **Analyst's Note**.
>
> In **150 words or fewer**, tell her what you found and what you recommend.
>
> **Banned words:** *accuracy*, *recall*, *precision*, *AUC*, *threshold*. If you need the idea,
> say it in her language.
>
> Your note has to make clear (i) what the model can do for her team, (ii) what it cannot do, and
> (iii) what you would need in order to do better.

---

## Before you submit

1. **Run this notebook top to bottom** with *Runtime → Restart and run all*, and check it finishes
   without errors.
2. **Share → General access → Anyone with the link.** Test it in a private window.
3. Open **Week 6 Homework - Model Evaluation** in Canvas and paste the link into the first question.
4. Answer (a) through (e) and the Analyst's Note **in Canvas**. Nothing in this notebook is read.